# Evaluate a multimodal library evidence fixture

Use synthetic image vectors, transcript segments and outcome labels. No media service or download is needed.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Route a media request

Route requests to representations that retain the needed evidence.

Implement choose_input(media, need_layout, has_transcript). Text always uses text. For image and audio, use transcript if available and need_layout is false, otherwise use the original modality. Here need_layout means the task needs spatial, temporal or other nontext evidence. Reject unknown media.


In [ ]:
def choose_input(media, need_layout, has_transcript):
    raise NotImplementedError


In [ ]:
assert choose_input("text",True,False) == "text"
assert choose_input("text",False,True) == "text", "Text always uses text"
assert choose_input("image",False,True) == "transcript"
assert choose_input("image",True,True) == "image"
assert choose_input("audio",False,False) == "audio"
try:
    choose_input("other",False,True)
except ValueError:
    pass
else:
    raise AssertionError("Reject unrecognized media")
print("OK")


## 2. Rank aligned image vectors

Rank image results using a compatible text query vector.

Implement rank_images(query, images). Each image has a distinct ID and a unit embedding from the aligned space. Sort by decreasing dot product then ascending ID. Reject dimension mismatch.


In [ ]:
def rank_images(query, images):
    raise NotImplementedError


In [ ]:
assert rank_images([1,0],[("z",[1,0]),("a",[1,0]),("map",[0,1])]) == ["a","z","map"]
assert rank_images([0,1],[("z",[1,0]),("map",[0,1])]) == ["map","z"]
assert rank_images([1],[]) == []
try:
    rank_images([1,0],[("bad",[1])])
except ValueError:
    pass
else:
    raise AssertionError("Reject mismatched dimensions")
print("OK")


## 3. Locate a transcript segment

Find the source transcript segment for a citation.

Implement transcript_segment(segments, time) for ordered, nonoverlapping (start, exclusive_end, text) rows in seconds. Return the matching text or None. Preserve empty segment text.


In [ ]:
def transcript_segment(segments, time):
    raise NotImplementedError


In [ ]:
segments = [(0,2,"one"),(2,4,"two"),(6,7,"")]
assert transcript_segment(segments,0) == "one"
assert transcript_segment(segments,2) == "two"
assert transcript_segment(segments,4) is None
assert transcript_segment(segments,6) == ""
assert transcript_segment(segments,7) is None
print("OK")


## 4. Count outcomes by media condition

Summarize supported, unsupported and abstained outcomes across conditions.

Implement modality_outcomes(rows). Rows are (condition, outcome) pairs. Return a nested count dictionary with only observed conditions and outcomes.


In [ ]:
def modality_outcomes(rows):
    raise NotImplementedError


In [ ]:
assert modality_outcomes([("scan","pass"),("scan","fail"),("scan","pass"),("audio","fail")]) == {"scan":{"pass":2,"fail":1},"audio":{"fail":1}}
assert modality_outcomes(iter([("scan","abstained")])) == {"scan":{"abstained":1}}
assert modality_outcomes([]) == {}
print("OK")


## 5. Evaluate timestamped evidence on paired conditions

Implement evidence_outcomes(cases, transcripts). Each case is (condition, media_id, time,
expected_text). The transcript mapping stores available segments by media ID. Missing media
have no segments. A missing observation is "abstained" if expected_text is None (correctly
declining when there is no answer) and "missed" otherwise (the evidence should have been found).
A present observation is "supported" only if expected_text is not None and equals the observed
text. Every other present observation is "unsupported". Return nested counts using
modality_outcomes.


In [ ]:
def evidence_outcomes(cases, transcripts):
    raise NotImplementedError


In [ ]:
cases = [("clean","meeting",2,"nine"),("noisy","missing",2,"nine"),
         ("noisy","noisy-copy",2,"nine"),("clean","meeting",9,None)]
transcripts = {"meeting":[(0,3,"nine")],"noisy-copy":[(0,3,"five")]}
assert evidence_outcomes(cases,transcripts) == {"clean":{"supported":1,"abstained":1},"noisy":{"missed":1,"unsupported":1}}
assert evidence_outcomes([("clean","empty",1,"")],{"empty":[(0,2,"")]}) == {"clean":{"supported":1}}
assert evidence_outcomes([],{}) == {}
print("OK")


## Interpretation

Describe one task where OCR should suffice and one that requires the source image. Specify paired clean/degraded cases and a criterion for abstention that an evaluator could apply.

**Your answer**
